# Error Analysis

Compares Majority Baseline, NBOW+LR, and DistilBERT systematically.

**Prerequisites:** NBOW and distilBERT notebooks fully run.

**Outputs produced:**
- `results/errors_nbow_fp/fn.csv`, `results/errors_bert_fp/fn.csv` — annotation files
- `results/model_outcome_comparison.png` — FP/FN/correct counts across all three models
- `results/error_category_fn/fp.png` — category breakdown charts
- `results/model_comparison.png` — final metric comparison

In [1]:
import sys, os, json
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.error_analysis import (
    CATEGORY_DEFINITIONS,
    merge_predictions,
    error_overlap_summary,
    prepare_annotation_files,
    load_annotated,
    compute_agreement,
    category_breakdown,
    plot_category_breakdown,
    plot_model_comparison_bars,
    build_results_table,
)
from src.evaluation import plot_all_models

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
os.makedirs('../results', exist_ok=True)
print("Imports OK")


Imports OK


## 1. Category definitions

Read and agree on these **before** looking at any predictions.

In [2]:
print("=== ERROR CATEGORY DEFINITIONS ===\n")
for key, defn in CATEGORY_DEFINITIONS.items():
    print(f"  {key:<22} {defn}")


=== ERROR CATEGORY DEFINITIONS ===

  sarcasm                Toxic in intent but no overtly toxic surface words. Irony or mock-politeness.
  non_toxic_profanity    Profanity used casually or affectionately, not directed at anyone.
  ambiguous              Could be toxic or non-toxic without additional context.
  discusses_hate         Describes or quotes hateful content without being hateful itself.
  implicit_toxicity      Toxic through implication, condescension, or dog-whistles. No explicit slurs.
  other                  Does not fit the above. Use sparingly.


## 2. Merge all predictions

In [3]:
df = merge_predictions(
    test_path='../data/test_split.csv',
    nbow_path='../results/test_nbow_lr_preds.csv',
    bert_path='../results/test_distilbert_preds.csv',
)


FileNotFoundError: [Errno 2] No such file or directory: '../results/test_distilbert_preds.csv'

## 3. Three-way outcome comparison

How do all three models compare in terms of raw correct/incorrect counts?

In [ ]:
overlap = error_overlap_summary(df)
print(overlap.to_string(index=False))


In [ ]:
# Visual comparison: FP / FN / correct counts per model
plot_model_comparison_bars(df, save=True)

In [ ]:
# The majority baseline has zero false positives and all toxic comments as false negatives.
# This is the floor every model must beat on recall.
n_toxic = df['label'].sum()
majority_fn = (df['majority_error'] == 'false_negative').sum()
print(f"Majority baseline misses {majority_fn:,} of {n_toxic:,} toxic comments (100% FN rate on toxic class)")
print()

# How many toxic comments does each model still miss?
for model, col in [("NBOW + LR", "nbow_error"), ("DistilBERT", "bert_error")]:
    fn = (df[col] == 'false_negative').sum()
    fp = (df[col] == 'false_positive').sum()
    print(f"{model}:")
    print(f"  Still misses {fn:,} of {n_toxic:,} toxic comments  ({fn/n_toxic*100:.1f}% FN rate)")
    print(f"  False alarms: {fp:,} non-toxic comments flagged")
    print()


## 4. Model agreement analysis

In [ ]:
total = len(df)

for model_a, col_a, model_b, col_b in [
    ("NBOW+LR", "nbow_error", "DistilBERT", "bert_error"),
]:
    only_a = ((df[col_a]=='correct') & (df[col_b]!='correct')).sum()
    only_b = ((df[col_b]=='correct') & (df[col_a]!='correct')).sum()
    both   = ((df[col_a]=='correct') & (df[col_b]=='correct')).sum()
    neither= ((df[col_a]!='correct') & (df[col_b]!='correct')).sum()

    print(f"Only {model_a} correct:   {only_a:,}  ({only_a/total*100:.1f}%)")
    print(f"Only {model_b} correct:  {only_b:,}  ({only_b/total*100:.1f}%)")
    print(f"Both correct:            {both:,}  ({both/total*100:.1f}%)")
    print(f"Neither correct:         {neither:,}  ({neither/total*100:.1f}%)")


In [ ]:
bert_recovers = df[
    (df['nbow_error'] == 'false_negative') &
    (df['bert_error'] == 'correct')
]
print(f"Toxic comments NBOW misses but DistilBERT catches: {len(bert_recovers):,}")
print()
pd.set_option('display.max_colwidth', 250)
print("Sample:")
for _, row in bert_recovers.sample(min(6, len(bert_recovers)), random_state=42).iterrows():
    print(f"  nbow_prob={row['nbow_lr_prob']:.3f}  bert_prob={row['distilbert_prob']:.3f}")
    print(f"  {row['comment_text'][:220]}")
    print()


In [ ]:
# Where does NBOW do better? Toxic comments BERT misses but NBOW catches.
nbow_recovers = df[
    (df['bert_error'] == 'false_negative') &
    (df['nbow_error'] == 'correct')
]
print(f"Toxic comments DistilBERT misses but NBOW catches: {len(nbow_recovers):,}")
print()
for _, row in nbow_recovers.sample(min(6, len(nbow_recovers)), random_state=42).iterrows():
    print(f"  nbow_prob={row['nbow_lr_prob']:.3f}  bert_prob={row['distilbert_prob']:.3f}")
    print(f"  {row['comment_text'][:220]}")
    print()


In [ ]:
# Comments that fool all three models (hardest cases)
hard_cases = df[df['all_wrong'] & (df['label'] == 1)]  # toxic, all models missed
print(f"Toxic comments all three models miss: {len(hard_cases):,}")
print()
for _, row in hard_cases.sample(min(8, len(hard_cases)), random_state=42).iterrows():
    print(f"  nbow={row['nbow_lr_prob']:.3f}  bert={row['distilbert_prob']:.3f}")
    print(f"  {row['comment_text'][:220]}")
    print()


## 5. Prepare annotation files

In [ ]:
prepare_annotation_files(df, n_samples=50)
print("\nOpen the CSVs in results/ and fill in the 'category' column independently.")


## 6. Inter-annotator agreement

Run after both annotators have independently labelled the same file.

In [ ]:
# Example — uncomment once annotation is complete
# kappa = compute_agreement(
#     '../results/errors_nbow_fn_annotator_a.csv',
#     '../results/errors_nbow_fn_annotator_b.csv',
# )
print("Uncomment the above once annotation is complete.")
print("Target kappa > 0.6.  If < 0.4, revisit category definitions.")


## 7. Category breakdown

After annotation and disagreement resolution, load the final labeled files.

In [ ]:
try:
    nbow_fp = load_annotated('../results/errors_nbow_fp.csv')
    nbow_fn = load_annotated('../results/errors_nbow_fn.csv')
    bert_fp = load_annotated('../results/errors_bert_fp.csv')
    bert_fn = load_annotated('../results/errors_bert_fn.csv')
    annotation_done = True
    print("Annotation files loaded.")
except Exception as e:
    print(f"Not yet annotated: {e}")
    annotation_done = False


In [ ]:
if annotation_done:
    # False negatives — toxic comments each model missed
    breakdown_fn = category_breakdown({
        'NBOW FN': nbow_fn,
        'BERT FN': bert_fn,
    })
    print("False negative category breakdown:")
    print(breakdown_fn.pivot_table(index='category', columns='label', values='pct', fill_value=0).to_string())
    print()
    plot_category_breakdown(breakdown_fn, title="False negatives — toxic comments missed", save=True)


In [ ]:
if annotation_done:
    # False positives — non-toxic comments each model flagged
    breakdown_fp = category_breakdown({
        'NBOW FP': nbow_fp,
        'BERT FP': bert_fp,
    })
    print("False positive category breakdown:")
    print(breakdown_fp.to_string(index=False))
    print()
    plot_category_breakdown(breakdown_fp, title="False positives — non-toxic flagged as toxic", save=True)


## 8. Qualitative examples per category

In [ ]:
if annotation_done:
    # For each category, show examples from both models side by side
    # so you can see whether they fail for the same reason
    for category in CATEGORY_DEFINITIONS.keys():
        nbow_examples = nbow_fn[nbow_fn['category'] == category]
        bert_examples = bert_fn[bert_fn['category'] == category]
        if len(nbow_examples) == 0 and len(bert_examples) == 0:
            continue

        print(f"\n{'='*65}")
        print(f"  Category: {category}  |  NBOW FN: {len(nbow_examples)}  |  BERT FN: {len(bert_examples)}")
        print(f"{'='*65}")

        print("  NBOW misses:")
        for _, row in nbow_examples.head(2).iterrows():
            print(f"    prob={row['nbow_lr_prob']:.3f}  {str(row['comment_text'])[:200]}")
        print()
        print("  BERT misses:")
        for _, row in bert_examples.head(2).iterrows():
            print(f"    prob={row['distilbert_prob']:.3f}  {str(row['comment_text'])[:200]}")
        print()


In [ ]:
if annotation_done:
    # Summary table: for each category, how many errors does each model make?
    summary_rows = []
    for cat in CATEGORY_DEFINITIONS.keys():
        summary_rows.append({
            'category':    cat,
            'nbow_fn':     (nbow_fn['category'] == cat).sum(),
            'bert_fn':     (bert_fn['category'] == cat).sum(),
            'nbow_fp':     (nbow_fp['category'] == cat).sum(),
            'bert_fp':     (bert_fp['category'] == cat).sum(),
        })
    summary = pd.DataFrame(summary_rows)
    print("Error counts by category and model:")
    print(summary.to_string(index=False))


## 9. Final metric comparison — all three models

In [ ]:
try:
    results_table = build_results_table(results_dir=Path('../results'))
    print("=== Final results table (sorted by toxic F1) ===")
    display(results_table)
except FileNotFoundError as e:
    print(f"Missing result files: {e}")


In [ ]:
try:
    plot_all_models(results_dir=Path('../results'))
except Exception as e:
    print(f"Could not plot: {e}")


In [ ]:
# How much does each model improve over the majority baseline?
# Compute this from the merged predictions directly (no JSON needed)
from sklearn.metrics import f1_score

y_true = df['label'].values

for model, pred_col in [
    ("Majority Baseline", "majority_pred"),
    ("NBOW + LR",         "nbow_lr_pred"),
    ("DistilBERT",        "distilbert_pred"),
]:
    f1  = f1_score(y_true, df[pred_col].values, pos_label=1, zero_division=0)
    print(f"{model:<22}  F1 (toxic): {f1:.4f}")
